In [ ]:
%pip install -U ipywidgets

In [ ]:
%pip install git+https://github.com/py-why/causal-learn.git

  Cloning https://github.com/py-why/causal-learn.git to c:\users\hp\appdata\local\temp\pip-req-build-ghfbp0wi
  Resolved https://github.com/py-why/causal-learn.git to commit 9689c1bdc468847729eacf0921b76f598161ae16
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'


  Running command git clone --filter=blob:none --quiet https://github.com/py-why/causal-learn.git 'C:\Users\HP\AppData\Local\Temp\pip-req-build-ghfbp0wi'


In [1]:
import numpy as np
import pandas as pd
import scipy
import os
import time

In [4]:
givenParams = ['WOW', 'LATP', 'LONP', 'BAL1', 'ROLL', 'PTCH', 'IVV', 'GS', 'CAS', 'VRTG', 'LATG', 'FLAP', 'PLA_2', 'PLA_3', 'PLA_4', 'TRK', 'TH', 'WS', 'WD', 'TAT', 'SAT', 'LOC', 'GLS']
var_names = [f"{param}_mean" for param in givenParams] + [f"{param}_std" for param in givenParams] + ["isLongLanding"]

df = pd.read_csv('FullFilteredRequiredData.csv')
df.drop(columns=['TouchdownDistance'], inplace=True)
df.head()

,WOW_mean,LATP_mean,LONP_mean,BAL1_mean,ROLL_mean,PTCH_mean,IVV_mean,GS_mean,CAS_mean,VRTG_mean,...,PLA_4_std,TRK_std,TH_std,WS_std,WD_std,TAT_std,SAT_std,LOC_std,GLS_std,isLongLanding
0,0.952381,46.907830,-96.815466,926.583333,0.308523,0.350309,-429.431548,110.041667,116.068452,1.009552,...,4.292836,0.237390,0.374503,0.861338,7.157449,1.022008,1.213352,0.003225,0.169581,0.0
1,0.952381,35.058233,-89.953914,320.095238,-1.124300,-0.782557,-501.000000,119.778274,120.196429,1.001922,...,8.545955,0.578280,2.304534,4.595623,63.336208,0.976916,1.244717,0.012306,0.070444,0.0
2,0.952381,42.227989,-83.365766,704.011905,0.409032,0.227174,-422.184524,116.321429,116.225446,1.000219,...,9.463613,0.500724,1.202374,2.575892,7.046211,1.062396,1.218829,0.001691,0.234278,0.0
3,0.952381,38.042018,-84.596677,1043.773810,-0.065328,0.299499,-360.699405,113.281250,113.395833,0.790925,...,8.860344,0.377983,1.607919,2.038931,149.417796,1.128381,1.235918,0.015060,0.295813,0.0
4,0.952381,41.538736,-93.651606,977.035714,-0.348021,-0.642289,-349.883929,117.413690,119.049107,1.002454,...,8.689983,0.219237,1.207223,3.624479,98.996618,0.929944,1.218829,0.026006,0.151445,0.0


### Using the __LINGAM__ Algorithm (removing __outliers__)

In [3]:
from causalai.models.tabular.lingam import LINGAM

from causalai.data.data_generator import DataGenerator
# also importing data object, data transform object, and prior knowledge object, and the graph plotting function
from causalai.data.tabular import TabularData
from causalai.data.transforms.tabular import StandardizeTransform
from causalai.models.common.prior_knowledge import PriorKnowledge
from causalai.misc.misc import plot_graph, get_precision_recall, get_precision_recall_skeleton, make_symmetric

In [5]:
# 1.
StandardizeTransform_ = StandardizeTransform()
StandardizeTransform_.fit(df.to_numpy())

data_trans = StandardizeTransform_.transform(df.to_numpy())

# 2.
data_obj = TabularData(data_trans, var_names=var_names)

In [6]:
prior_knowledge = None

lingam = LINGAM(
        data=data_obj,
        prior_knowledge=prior_knowledge,
        )
tic = time.time()
result = lingam.run(pvalue_thres=0.05)
toc = time.time()
print(f'Time taken: {toc-tic:.2f}s\n')


print(f' The output result has keys: {result.keys()}')
print(f' The output result["WOW_mean"] has keys: {result["WOW_mean"].keys()}')

Time taken: 286.75s

 The output result has keys: dict_keys(['WOW_mean', 'LATP_mean', 'LONP_mean', 'BAL1_mean', 'ROLL_mean', 'PTCH_mean', 'IVV_mean', 'GS_mean', 'CAS_mean', 'VRTG_mean', 'LATG_mean', 'FLAP_mean', 'PLA_2_mean', 'PLA_3_mean', 'PLA_4_mean', 'TRK_mean', 'TH_mean', 'WS_mean', 'WD_mean', 'TAT_mean', 'SAT_mean', 'LOC_mean', 'GLS_mean', 'WOW_std', 'LATP_std', 'LONP_std', 'BAL1_std', 'ROLL_std', 'PTCH_std', 'IVV_std', 'GS_std', 'CAS_std', 'VRTG_std', 'LATG_std', 'FLAP_std', 'PLA_2_std', 'PLA_3_std', 'PLA_4_std', 'TRK_std', 'TH_std', 'WS_std', 'WD_std', 'TAT_std', 'SAT_std', 'LOC_std', 'GLS_std', 'isLongLanding'])
 The output result["WOW_mean"] has keys: dict_keys(['value_dict', 'pvalue_dict', 'parents'])


In [7]:
print(f'Predicted parents:')
result['isLongLanding']['parents']

Predicted parents:


['FLAP_std']

__Causal Strengths__

In [8]:
causalStrengthsDictLINGAM = {k:result['isLongLanding']['value_dict'][k] for k in result['isLongLanding']['parents']}
causalStrengthsDictLINGAM

{'FLAP_std': np.float64(0.02039903647220258)}

In [9]:
# Sorting the dictionary by absolute value in descending order
sorted_dict = dict(sorted(causalStrengthsDictLINGAM.items(), key=lambda item: abs(item[1]), reverse=True))

# Printing the sorted dictionary
sorted_dict

{'FLAP_std': np.float64(0.02039903647220258)}

__Statistical Significance__

In [10]:
statSignDictLINGAM = {k:result['isLongLanding']['pvalue_dict'][k] for k in result['isLongLanding']['parents']}
statSignDictLINGAM

{'FLAP_std': np.float64(0.0005318810988922126)}

In [11]:
# Sort by ascending p-value
sorted_pvalues = dict(sorted(statSignDictLINGAM.items(), key=lambda x: x[1]))

# Display
sorted_pvalues

{'FLAP_std': np.float64(0.0005318810988922126)}

In [12]:
documentationDict = {}
with open("variable_documentation.txt", "r") as f:
    for line in f:
        varName, desc = line.split(':')
        documentationDict[varName.strip()] = desc.strip()

In [13]:
mainVars = set(['_'.join(v.split('_')[:-1]) for v in result['isLongLanding']['parents']])
for var in mainVars:
    print(f"{var}: {documentationDict[var]}")

FLAP: T.E. FLAP POSITION


### Using the __PC__ Algorithm (removing __outliers__)

In [14]:
from causalai.models.tabular.pc import PCSingle, PC
from causalai.models.common.CI_tests.partial_correlation import PartialCorrelation
from causalai.models.common.CI_tests.discrete_ci_tests import DiscreteCI_tests
from causalai.models.common.CI_tests.kci import KCI


# also importing data object, data transform object, and prior knowledge object, and the graph plotting function
from causalai.data.data_generator import DataGenerator, GenerateRandomTabularSEM
from causalai.data.tabular import TabularData
from causalai.data.transforms.time_series import StandardizeTransform
from causalai.models.common.prior_knowledge import PriorKnowledge
from causalai.misc.misc import plot_graph, get_precision_recall, get_precision_recall_skeleton, make_symmetric

In [15]:
from causallearn.search.ConstraintBased.PC import pc
from causallearn.utils.GraphUtils import GraphUtils
from causallearn.utils.cit import fisherz

In [16]:
# 1.
StandardizeTransform_ = StandardizeTransform()
StandardizeTransform_.fit(df.to_numpy())

data_trans = StandardizeTransform_.transform(df.to_numpy())

# 2.
data_obj = TabularData(data_trans, var_names=var_names)

In [17]:
prior_knowledge = None #  PriorKnowledge(forbidden_links={'a': ['b']})

pvalue_thres = 0.05
CI_test = PartialCorrelation()
# CI_test = KCI(chunk_size=100) # use if the causal relationship is expected to be non-linear
pc = PC(
        data=data_obj,
        prior_knowledge=prior_knowledge,
        CI_test=CI_test,
        use_multiprocessing=True
        )

In [30]:
tic = time.time()


result = pc.run(pvalue_thres=pvalue_thres, max_condition_set_size=2)

toc = time.time()

print(f'Time taken: {toc-tic:.2f}s\n')

print(f' The output result has keys: {result.keys()}')
print(f' The output result["WOW_mean"] has keys: {result["WOW_mean"].keys()}')

2025-05-03 10:48:50,086	INFO worker.py:1852 -- Started a local Ray instance.


Time taken: 811.18s

 The output result has keys: dict_keys(['WOW_mean', 'LATP_mean', 'LONP_mean', 'BAL1_mean', 'ROLL_mean', 'PTCH_mean', 'IVV_mean', 'GS_mean', 'CAS_mean', 'VRTG_mean', 'LATG_mean', 'FLAP_mean', 'PLA_2_mean', 'PLA_3_mean', 'PLA_4_mean', 'TRK_mean', 'TH_mean', 'WS_mean', 'WD_mean', 'TAT_mean', 'SAT_mean', 'LOC_mean', 'GLS_mean', 'WOW_std', 'LATP_std', 'LONP_std', 'BAL1_std', 'ROLL_std', 'PTCH_std', 'IVV_std', 'GS_std', 'CAS_std', 'VRTG_std', 'LATG_std', 'FLAP_std', 'PLA_2_std', 'PLA_3_std', 'PLA_4_std', 'TRK_std', 'TH_std', 'WS_std', 'WD_std', 'TAT_std', 'SAT_std', 'LOC_std', 'GLS_std', 'isLongLanding'])
 The output result["WOW_mean"] has keys: dict_keys(['parents', 'value_dict', 'pvalue_dict'])


In [31]:
result['isLongLanding']['parents']

['BAL1_std',
 'FLAP_mean',
 'GS_std',
 'GLS_std',
 'GS_mean',
 'TH_mean',
 'TAT_std',
 'IVV_mean',
 'LONP_std',
 'CAS_std',
 'LOC_std']

__Causal Strengths__

In [32]:
# Sorting the dictionary by absolute value in descending order
sorted_dict = dict(sorted(result['isLongLanding']['value_dict'].items(), key=lambda item: abs(item[1]), reverse=True))

# Printing the sorted dictionary
sorted_dict

{'GS_std': np.float64(0.06157252918014551),
 'BAL1_std': np.float64(-0.03997386641744888),
 'TH_mean': np.float64(0.03695598339425889),
 'LOC_std': np.float64(0.0353544104926993),
 'FLAP_mean': np.float64(-0.028715136004050387),
 'GLS_std': np.float64(-0.024020678239888012),
 'CAS_std': np.float64(0.019471877383699064),
 'GS_mean': np.float64(0.016900635814833483),
 'LONP_std': np.float64(0.015505298878732341),
 'TAT_std': np.float64(0.013853217207236496),
 'IVV_mean': np.float64(0.009636058942403665)}

__Statistical Significance__

In [33]:
# Assuming this is your pvalue_dict:
pvalue_dict = result['isLongLanding']['pvalue_dict']

# Sort by ascending p-value
sorted_pvalues = dict(sorted(pvalue_dict.items(), key=lambda x: x[1]))

# Display
sorted_pvalues

{'GS_std': np.float64(1.0432467513843412e-39),
 'BAL1_std': np.float64(1.1637330925841474e-17),
 'TH_mean': np.float64(2.571479857327294e-15),
 'LOC_std': np.float64(3.8190830864626396e-14),
 'FLAP_mean': np.float64(7.979577458194045e-10),
 'GLS_std': np.float64(2.742229269928708e-07),
 'CAS_std': np.float64(3.0901768776963904e-05),
 'GS_mean': np.float64(0.0002987035999299433),
 'LONP_std': np.float64(0.0009071504192900701),
 'TAT_std': np.float64(0.003033785645657922),
 'IVV_mean': np.float64(0.03922316225970596)}

In [34]:
documentationDict = {}
with open("variable_documentation.txt", "r") as f:
    for line in f:
        varName, desc = line.split(':')
        documentationDict[varName.strip()] = desc.strip()

In [35]:
mainVars = set(['_'.join(v.split('_')[:-1]) for v in result['isLongLanding']['parents']])
for var in mainVars:
    print(f"{var}: {documentationDict[var]}")

LOC: LOCALIZER DEVIATION
BAL1: BARO CORRECT ALTITUDE LSP
IVV: INERTIAL VERTICAL SPEED LSP
CAS: COMPUTED AIRSPEED LSP
TH: TRUE HEADING LSP
GS: GROUND SPEED LSP
LONP: LONGITUDE POSITION LSP
TAT: TOTAL AIR TEMPERATURE
FLAP: T.E. FLAP POSITION
GLS: GLIDESLOPE DEVIATION


### Using the __Grow-Shrink__ Algorithm (removing __outliers__)

In [36]:
from causalai.models.tabular.grow_shrink import GrowShrink
from causalai.models.common.CI_tests.partial_correlation import PartialCorrelation
from causalai.models.common.CI_tests.discrete_ci_tests import DiscreteCI_tests
from causalai.models.common.CI_tests.kci import KCI


# also importing data object, and prior knowledge object, and the graph plotting function
from causalai.data.data_generator import DataGenerator, GenerateRandomTabularSEM
from causalai.data.tabular import TabularData
from causalai.data.transforms.time_series import StandardizeTransform
from causalai.models.common.prior_knowledge import PriorKnowledge
from causalai.misc.misc import plot_graph, get_precision_recall, get_precision_recall_skeleton, make_symmetric,\
                               _get_precision_recall_single

# Helper functions to compute ground truth

from causalai.tests.data.transforms.networkx_helper_functions import causalai2networkx, compute_markov_blanket

In [37]:
# 1.
StandardizeTransform_ = StandardizeTransform()
StandardizeTransform_.fit(df.to_numpy())

data_trans = StandardizeTransform_.transform(df.to_numpy())

# 2.
data_obj = TabularData(data_trans, var_names=var_names)

In [38]:
prior_knowledge = None #  PriorKnowledge(forbidden_links={'a': ['b']})

pvalue_thres = 0.05
CI_test = PartialCorrelation()
# CI_test = KCI(chunk_size=100) # use if the causal relationship is expected to be non-linear
gs = GrowShrink(
        data=data_obj,
        prior_knowledge=prior_knowledge,
        CI_test=CI_test,
        use_multiprocessing=True,
        update_shrink=False
        )

target_var = 'isLongLanding'

In [39]:
tic = time.time()


result = gs.run(target_var=target_var, pvalue_thres=pvalue_thres)

toc = time.time()

print(f'Time taken: {toc-tic:.2f}s\n')

print(f" target var {target_var}\'s estimated MB is: {result['markov_blanket']}")

2025-05-03 11:02:21,486	INFO worker.py:1852 -- Started a local Ray instance.


Time taken: 13.46s

 target var isLongLanding's estimated MB is: ['LATP_std', 'GS_mean', 'TAT_mean', 'TAT_std', 'WOW_mean', 'TH_mean', 'CAS_std', 'LOC_std', 'BAL1_std', 'IVV_std', 'BAL1_mean', 'GS_std', 'GLS_std', 'CAS_mean', 'PTCH_std', 'FLAP_std', 'WOW_std', 'SAT_std', 'LONP_std', 'WD_mean', 'WS_std', 'FLAP_mean', 'SAT_mean', 'LATP_mean', 'WS_mean', 'LONP_mean']


In [40]:
result['markov_blanket']

['LATP_std',
 'GS_mean',
 'TAT_mean',
 'TAT_std',
 'WOW_mean',
 'TH_mean',
 'CAS_std',
 'LOC_std',
 'BAL1_std',
 'IVV_std',
 'BAL1_mean',
 'GS_std',
 'GLS_std',
 'CAS_mean',
 'PTCH_std',
 'FLAP_std',
 'WOW_std',
 'SAT_std',
 'LONP_std',
 'WD_mean',
 'WS_std',
 'FLAP_mean',
 'SAT_mean',
 'LATP_mean',
 'WS_mean',
 'LONP_mean']

__Causal Strengths__

In [41]:
# Extract variable names and their causal strengths
var_strength_pairs = [(var, result['value_dict'][var]) for var in result['markov_blanket']]

# Sort based on absolute value of causal strength, descending
var_strength_pairs_sorted = sorted(var_strength_pairs, key=lambda x: abs(x[1]), reverse=True)

# Print the sorted results
for var, strength in var_strength_pairs_sorted:
    print(f"{var}: {strength:.4f}")

BAL1_std: -0.0750
WOW_std: -0.0564
GS_mean: 0.0550
WOW_mean: -0.0499
TH_mean: 0.0482
GLS_std: -0.0476
GS_std: 0.0451
LOC_std: 0.0361
LATP_std: -0.0309
WS_std: 0.0299
LATP_mean: 0.0267
IVV_std: 0.0237
BAL1_mean: -0.0215
PTCH_std: -0.0215
CAS_mean: -0.0195
TAT_std: 0.0193
SAT_mean: -0.0165
TAT_mean: 0.0164
WD_mean: -0.0160
LONP_mean: 0.0158
FLAP_std: 0.0158
CAS_std: -0.0144
LONP_std: -0.0133
SAT_std: -0.0102
WS_mean: 0.0096
FLAP_mean: -0.0095


__Statistical Significance__

In [42]:
# Extract variable names and their p-values
var_pval_pairs = [(var, result['pvalue_dict'][var]) for var in result['markov_blanket']]

# Sort based on p-value in ascending order
var_pval_pairs_sorted = sorted(var_pval_pairs, key=lambda x: x[1])

# Print the sorted results
for var, pval in var_pval_pairs_sorted:
    print(f"{var}: {pval}")

BAL1_std: 4.9030586333262436e-58
WOW_std: 1.4323462757061154e-33
GS_mean: 5.540074609002877e-32
WOW_mean: 1.409348773787862e-26
TH_mean: 5.604905504046876e-25
GLS_std: 2.3429671706500372e-24
GS_std: 4.590254358844617e-22
LOC_std: 1.1075210228620108e-14
LATP_std: 4.0799788768976357e-11
WS_std: 1.5148081870558315e-10
LATP_mean: 1.0670359478472146e-08
IVV_std: 3.903990112692036e-07
BAL1_mean: 4.0491835482178765e-06
PTCH_std: 4.4068632364735e-06
CAS_mean: 3.129499098003772e-05
TAT_std: 3.716072027743374e-05
SAT_mean: 0.00040477101158583245
TAT_mean: 0.0004415848582198509
WD_mean: 0.000643388508681758
LONP_mean: 0.0007036988116947664
FLAP_std: 0.0007322567170888239
CAS_std: 0.0021268544584486697
LONP_std: 0.00444349486013873
SAT_std: 0.029754496439911998
WS_mean: 0.04028463589454426
FLAP_mean: 0.04187476418800186


## Understanding Conditional Independence and p-values in Causal Discovery

### Scenario:
Suppose we have three variables: **A**, **B**, and **C**. We're trying to understand the causal relationship between **A** and **C**, while considering **B**.

---

### The Null Hypothesis (H₀):
> **A and C are conditionally independent given B**

This means:
> _“Once we know B, knowing A gives us no additional information about C.”_

---

### What does the p-value tell us?

The **p-value** helps assess how compatible our data is with the null hypothesis (H₀).  
In the context of causal discovery, it helps us decide whether the observed relationship between two variables could just be due to chance under the assumption of conditional independence.

---

### Interpreting p-values:

| p-value | Interpretation |
|--------|----------------|
| **Low (e.g., < 0.05)** | Strong evidence **against** H₀ → A and C are **not** conditionally independent given B → suggests a **potential causal relationship** |
| **High (e.g., > 0.05)** | Weak evidence against H₀ → A and C **may be** conditionally independent given B → B might **explain** the relationship between A and C |

---

### Practical Implication:

If the causal structure is:

A-->B-->C

Then A and C are **dependent** in general.  
But **once we condition on B**, A and C become **independent** — there's **no direct path** from A to C.

This principle helps algorithms like **LiNGAM** discover and remove **indirect or spurious causal links**, keeping only the **true direct causal influences**.

---

### 📌 Summary:
- **p-value** evaluates whether two variables are **conditionally independent** given others.
- A **low p-value** → evidence of a **direct causal effect**.
- A **high p-value** → suggests the effect might be explained by **indirect paths** or **confounders**.